In [1]:
import os
import cv2
import torch
import pickle
import numpy as np
import torch.nn as nn
import mediapipe as mp
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score,accuracy_score

In [4]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Chuyển sang RGB
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                label = handedness.classification[0].label  # "Left" hoặc "Right"
                mp_drawing.draw_landmarks(
                    frame,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2, circle_radius=3),
                    mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2)
                )

                landmarks = []
                xs = []
                ys= []
                for lm in hand_landmarks.landmark:
                    x, y = lm.x, lm.y
                    if label == "Left":   # nếu là tay trái thì lật
                        x = 1 - x
                    landmarks.append((x, y))
                    if x !=0:
                        xs.append(x)
                    if y != 0:
                        ys.append(y)

                h, w, _ = frame.shape
                cx = int(hand_landmarks.landmark[0].x * w)
                cy = int(hand_landmarks.landmark[0].y * h)
                cv2.putText(frame, label, (cx - 50, cy - 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 0), 2, cv2.LINE_AA)

                if xs != [] and ys != []:
                    xmin, xmax = min(xs), max(xs)
                    ymin, ymax = min(ys), max(ys)
                    width, height = xmax - xmin, ymax - ymin

                    if width > height:
                        delta_x = 0.1 * width
                        delta_y = delta_x + (width - height) / 2
                    else:
                        delta_y = 0.1 * height
                        delta_x = delta_y + (height - width) / 2

                    start_x = max(int((xmin - delta_x) * w), 0)
                    start_y = max(int((ymin - delta_y) * h), 0)
                    end_x   = min(int((xmax + delta_x) * w), w)
                    end_y   = min(int((ymax + delta_y) * h), h)
                    landmarks_norm = []
                    for (x, y) in landmarks:
                        cx, cy = int(x * w), int(y * h)
                        x_norm = (cx - start_x) / (end_x - start_x)
                        y_norm = (cy - start_y) / (end_y - start_y)
                        landmarks_norm.append((x_norm, y_norm))

        cv2.imshow("Hands", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

In [3]:
CLASSES = ['E', 'Y', 'M', 'X', 'N', 'R', 'P', 'C', 'O', 'del', 'T', 
           'space', 'G', 'I', 'F', 'S', 'Q', 'V', 'K', 'U', 'A', 'B', 'L', 
           'W', 'D', 'H']

In [6]:
def bounding_box(width_frame, height_frame, x_landmarks, y_landmarks):
    
    xmin, xmax = min(x_landmarks), max(x_landmarks)
    ymin, ymax = min(y_landmarks), max(y_landmarks)
    width, height = xmax - xmin, ymax - ymin
    if width > height:
        delta_x = 0.1 * width
        delta_y = delta_x + (width - height) / 2
    else:
        delta_y = 0.1 * height
        delta_x = delta_y + (height - width) / 2
    start_x = max(int((xmin - delta_x) * width_frame), 0)
    start_y = max(int((ymin - delta_y) * height_frame), 0)
    end_x   = min(int((xmax + delta_x) * width_frame), width_frame)
    end_y   = min(int((ymax + delta_y) * height_frame), height_frame)
    landmarks_norm = []
    for i in range(len(x_landmarks)):
        x = x_landmarks[i]
        y = y_landmarks[i]
        cx, cy = int(x * width_frame), int(y * height_frame)
        x_norm = (cx - start_x) / (end_x - start_x)
        y_norm = (cy - start_y) / (end_y - start_y)
        landmarks_norm.append(x_norm)
        landmarks_norm.append(y_norm)
    
    result = {'landmarks': landmarks_norm,
              'bounding_box_start': (start_x, start_y),
              'bounding_box_end': (end_x, end_y)}
    return result

In [5]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

error_images= []

DATA = []
LABELS = []

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:
    
    for _class in CLASSES[:2]:
        for idx, img in enumerate(os.listdir(f'dataset/{_class}')[:10]):
            frame = cv2.imread(f'dataset/{_class}/{img}')
            if frame is None:
                print(f'Không đọc được ảnh: dataset/{_class}/{img}')
                error_images.append(f'dataset/{_class}/{img}')
                continue
            # Chuyển sang RGB
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(rgb)

            if results.multi_hand_landmarks and results.multi_handedness:
                for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                    label = handedness.classification[0].label  # "Left" hoặc "Right"

                    landmarks = []
                    xs = []
                    ys= []
                    for lm in hand_landmarks.landmark:
                        x, y = lm.x, lm.y
                        if label == "Left":   # nếu là tay trái thì lật
                            x = 1 - x
                        landmarks.append((x, y))
                        if x !=0:
                            xs.append(x)
                        if y != 0:
                            ys.append(y)

                    # Vẽ tay đã chuẩn hóa (tay nào cũng thành "Right")
                    h, w, _ = frame.shape

                    if xs != [] and ys != []:
                        xmin, xmax = min(xs), max(xs)
                        ymin, ymax = min(ys), max(ys)
                        width, height = xmax - xmin, ymax - ymin

                        if width > height:
                            delta_x = 0.1 * width
                            delta_y = delta_x + (width - height) / 2
                        else:
                            delta_y = 0.1 * height
                            delta_x = delta_y + (height - width) / 2

                        start_x = max(int((xmin - delta_x) * w), 0)
                        start_y = max(int((ymin - delta_y) * h), 0)
                        end_x   = min(int((xmax + delta_x) * w), w)
                        end_y   = min(int((ymax + delta_y) * h), h)
                        landmarks_norm = []
                        for (x, y) in landmarks:
                            cx, cy = int(x * w), int(y * h)
                            x_norm = (cx - start_x) / (end_x - start_x)
                            y_norm = (cy - start_y) / (end_y - start_y)
                            landmarks_norm.append((x_norm, y_norm))
                        DATA.append(landmarks_norm)
                        LABELS.append(_class)

            else:
                error_images.append(f'dataset/{_class}/{img}')
                print(f"Không phát hiện tay trong ảnh: dataset/{_class}/{img}")
                continue

cv2.destroyAllWindows()

with open('MLP_data.pickle', 'wb') as f:
    pickle.dump({'data': DATA, 'labels': LABELS}, f)

In [6]:
def loadDataset(filename: str):
    loaded = np.load(filename, allow_pickle=True).item()
    _data = loaded['data']
    _labels = loaded['labels']
    return _data, _labels

In [8]:
def train_model_2(data, labels):
    x_train, x_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, stratify=labels, random_state=42)

    model = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=1000, random_state=42)
    model.fit(x_train, y_train)
    y_pred = model.predict(x_val)

    print('Accuracy:', accuracy_score(y_val, y_pred) * 100, '%')
    # Đánh giá từng chỉ số
    precision = precision_score(y_val, y_pred, average='weighted')
    recall = recall_score(y_val, y_pred, average='weighted')
    f1 = f1_score(y_val, y_pred, average='weighted')

    print("🔸 Precision:", round(precision * 100, 2), "%")
    print("🔸 Recall:", round(recall * 100, 2), "%")
    print("🔸 F1 Score:", round(f1 * 100, 2), "%")

    # lưu model
    with open('MLP_model.p', 'wb') as f:
        pickle.dump(model, f)

In [36]:
loaded = np.load("dataset.npy", allow_pickle=True).item()
data = loaded["data"]
y = loaded["labels"]

print(data.shape)
print(y.shape)

train_model_2(data=data, labels=y)

(63110, 42)
(63110,)
Accuracy: 99.18396450641737 %
🔸 Precision: 99.19 %
🔸 Recall: 99.18 %
🔸 F1 Score: 99.18 %


In [8]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

model = pickle.load(open('MLP_model.p', 'rb'))

num_classes = len(model.classes_)

cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:

    while True:
        x_ = []
        y_ = []
        ret, frame = cap.read()
        H, W, _ = frame.shape
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)

        if results.multi_hand_landmarks and results.multi_handedness:
            # for hand_landmarks in results.multi_hand_landmarks:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                label = handedness.classification[0].label  # "Left" hoặc "Right"

                x_landmarks = []
                y_landmarks = []
                hand_data = []
                if label == "Left":
                    for lm in hand_landmarks.landmark:
                        x_landmarks.append(1- lm.x)
                        y_landmarks.append(lm.y)
                else:
                    for lm in hand_landmarks.landmark:
                        x_landmarks.append(lm.x)
                        y_landmarks.append(lm.y)

                if len(x_landmarks)+len(y_landmarks) == 42:
                    bounding_box_output = bounding_box(W, H, x_landmarks, y_landmarks)

                    landmarks_norm = bounding_box_output['landmarks']
                    (start_x, start_y) = bounding_box_output['bounding_box_start']

                    prediction = model.predict([np.array(landmarks_norm)])
                    probs = model.predict_proba([np.asarray(landmarks_norm)])[0]
                    current_class = prediction[0]

                    cv2.putText(frame, CLASSES[prediction[0]], (start_x, start_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        panel_height = max(H, num_classes * 30)
        output_panel = np.ones((panel_height, 250, 3), dtype=np.uint8) * 255

        if results.multi_hand_landmarks:
            label_probs = list(zip(model.classes_, probs))
            label_probs.sort(key=lambda x: x[1], reverse=True)

            for idx, (label, prob) in enumerate(label_probs):
                y_pos = 50 + idx * 24
                cv2.putText(output_panel, f"{label}: {prob * 100:.2f}%", (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1)

        frame_resized = cv2.resize(frame, (W, panel_height))
        combined = np.hstack((frame_resized, output_panel))
        cv2.imshow('Realtime Hand Detection', combined)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

In [5]:
cap.release()
cv2.destroyAllWindows()

In [9]:
class MyMLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.net(x)

In [10]:
def train_model_torch(data, labels, epochs=30, batch_size=64, lr=1e-3, device="cuda"):
    x_train, x_val, y_train, y_val = train_test_split(data, labels, test_size=0.2, stratify=labels, random_state=42)

    # Convert sang tensor
    x_train = torch.tensor(x_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    x_val   = torch.tensor(x_val, dtype=torch.float32)
    y_val   = torch.tensor(y_val, dtype=torch.long)

    train_ds = TensorDataset(x_train, y_train)
    val_ds   = TensorDataset(x_val, y_val)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size)

    model = MyMLP(x_train.shape[1], len(set(labels))).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb).argmax(dim=1)
                correct += (preds == yb).sum().item()
                total += yb.size(0)

        print(f"Epoch {epoch+1}/{epochs}, Val Accuracy: {correct/total:.2%}")

In [11]:
loaded = np.load("dataset.npy", allow_pickle=True).item()
data = loaded["data"]
y = loaded["labels"]

print(data.shape)
print(y.shape)

train_model_torch(data=data, labels=y)

(63110, 42)
(63110,)
Epoch 1/30, Val Accuracy: 83.08%
Epoch 2/30, Val Accuracy: 91.83%
Epoch 3/30, Val Accuracy: 94.37%
Epoch 4/30, Val Accuracy: 95.16%
Epoch 5/30, Val Accuracy: 96.20%
Epoch 6/30, Val Accuracy: 96.93%
Epoch 7/30, Val Accuracy: 97.23%
Epoch 8/30, Val Accuracy: 97.34%
Epoch 9/30, Val Accuracy: 97.58%
Epoch 10/30, Val Accuracy: 97.49%
Epoch 11/30, Val Accuracy: 98.04%
Epoch 12/30, Val Accuracy: 98.07%
Epoch 13/30, Val Accuracy: 98.17%
Epoch 14/30, Val Accuracy: 98.25%
Epoch 15/30, Val Accuracy: 98.19%
Epoch 16/30, Val Accuracy: 98.26%
Epoch 17/30, Val Accuracy: 98.27%
Epoch 18/30, Val Accuracy: 98.23%
Epoch 19/30, Val Accuracy: 98.35%
Epoch 20/30, Val Accuracy: 98.67%
Epoch 21/30, Val Accuracy: 98.55%
Epoch 22/30, Val Accuracy: 98.49%
Epoch 23/30, Val Accuracy: 98.59%
Epoch 24/30, Val Accuracy: 98.64%
Epoch 25/30, Val Accuracy: 98.67%
Epoch 26/30, Val Accuracy: 98.70%
Epoch 27/30, Val Accuracy: 98.63%
Epoch 28/30, Val Accuracy: 98.72%
Epoch 29/30, Val Accuracy: 98.68%
Ep

In [ ]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

model = pickle.load(open('MLP_model.p', 'rb'))

num_classes = len(model.classes_)

cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:

    while True:
        x_ = []
        y_ = []
        ret, frame = cap.read()
        H, W, _ = frame.shape
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(frame_rgb)

        if results.multi_hand_landmarks and results.multi_handedness:
            # for hand_landmarks in results.multi_hand_landmarks:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                label = handedness.classification[0].label  # "Left" hoặc "Right"

                x_landmarks = []
                y_landmarks = []
                hand_data = []
                if label == "Left":
                    for lm in hand_landmarks.landmark:
                        x_landmarks.append(1- lm.x)
                        y_landmarks.append(lm.y)
                else:
                    for lm in hand_landmarks.landmark:
                        x_landmarks.append(lm.x)
                        y_landmarks.append(lm.y)

                if len(x_landmarks)+len(y_landmarks) == 42:
                    bounding_box_output = bounding_box(W, H, x_landmarks, y_landmarks)

                    landmarks_norm = bounding_box_output['landmarks']
                    (start_x, start_y) = bounding_box_output['bounding_box_start']

                    prediction = model.predict([np.array(landmarks_norm)])
                    probs = model.predict_proba([np.asarray(landmarks_norm)])[0]
                    current_class = prediction[0]

                    cv2.putText(frame, CLASSES[prediction[0]], (start_x, start_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        panel_height = max(H, num_classes * 30)
        output_panel = np.ones((panel_height, 250, 3), dtype=np.uint8) * 255

        if results.multi_hand_landmarks:
            label_probs = list(zip(model.classes_, probs))
            label_probs.sort(key=lambda x: x[1], reverse=True)

            for idx, (label, prob) in enumerate(label_probs):
                y_pos = 50 + idx * 24
                cv2.putText(output_panel, f"{label}: {prob * 100:.2f}%", (10, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1)

        frame_resized = cv2.resize(frame, (W, panel_height))
        combined = np.hstack((frame_resized, output_panel))
        cv2.imshow('Realtime Hand Detection', combined)

        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()